# Diferencias en Diferencias — Python
## Econometría Avanzada — Ana María Díaz Escobar

**PARTE 1:** DiD básico 2×2 (equivalente a `base3.dta`)  
**PARTE 2:** DiD con múltiples periodos — datos equivalentes a `hospdd` (Stata)  
Prueba de tendencias paralelas y anticipación con `pyfixest`

```bash
pip install pyfixest pandas numpy matplotlib statsmodels
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import pyfixest as pf

np.random.seed(1234)
print('Paquetes cargados correctamente')

---
## PARTE 1: DiD BÁSICO (2 periodos)

Estructura equivalente a `base3.dta`:  
- **Resultado:** talla-para-edad (z-score)  
- **Tratamiento:** programa de nutrición (D=1)  
- **Periodos:** 0 = antes, 1 = después  
- **Efecto real:** +0.18 desviaciones estándar

**Recordatorio:** con solo 2 periodos, las pruebas formales de tendencias paralelas y anticipación son *imposibles* (requieren ≥ 2 periodos pre-tratamiento).

In [ ]:
# --- Simulación de datos (estructura base3.dta) ---
n_ind = 4000
efecto_real = 0.18

np.random.seed(1234)
D = np.random.binomial(1, 0.5, n_ind)  # tratamiento aleatorio

datos = pd.DataFrame({
    'id': np.repeat(np.arange(n_ind), 2),
    't':  np.tile([0, 1], n_ind),
    'D':  np.repeat(D, 2)
})

datos['y'] = (
    -0.5 +
    0.1  * datos['D'] +
    0.05 * datos['t'] +
    efecto_real * datos['D'] * datos['t'] +
    np.random.normal(0, 0.5, len(datos))
)

print(f'Observaciones: {len(datos)} ({n_ind} individuos × 2 periodos)')
print(f'Efecto real del programa: {efecto_real}')

In [ ]:
# --- Tabla 2×2 de medias ---
tabla = datos.groupby(['D', 't'])['y'].mean().unstack()
tabla.index = ['Controles (D=0)', 'Tratados (D=1)']
tabla.columns = ['Antes (t=0)', 'Después (t=1)']
tabla['Primera diferencia'] = tabla['Después (t=1)'] - tabla['Antes (t=0)']
print('Tabla 2×2:')
print(tabla.round(4))

# Estimador DiD manual
y_c0 = datos.loc[(datos['D']==0) & (datos['t']==0), 'y'].mean()
y_c1 = datos.loc[(datos['D']==0) & (datos['t']==1), 'y'].mean()
y_t0 = datos.loc[(datos['D']==1) & (datos['t']==0), 'y'].mean()
y_t1 = datos.loc[(datos['D']==1) & (datos['t']==1), 'y'].mean()

DD = (y_t1 - y_t0) - (y_c1 - y_c0)
print(f'\nEstimador DiD manual: {DD:.4f}  (esperado ~{efecto_real})')

In [ ]:
# --- Gráfico de tendencias (visualización del DiD, NO prueba de tendencias) ---
medias = datos.groupby(['D', 't'])['y'].mean().reset_index()
medias['grupo'] = medias['D'].map({0: 'Controles', 1: 'Tratados'})
medias['periodo'] = medias['t'].map({0: 'Antes', 1: 'Después'})

fig, ax = plt.subplots(figsize=(7, 5))
for grupo, color, marker in [('Tratados', '#1C3A6E', 'o'), ('Controles', '#8B1A1A', '^')]:
    d = medias[medias['grupo'] == grupo]
    ax.plot(d['periodo'], d['y'], marker=marker, color=color,
            linewidth=2, markersize=8, label=grupo)

ax.set_title('Evolución de la talla-para-edad por grupo', fontsize=13)
ax.set_xlabel('Periodo')
ax.set_ylabel('Talla-para-edad (z-score)')
ax.legend()
ax.text(0.5, -0.13,
        'Nota: con 2 periodos este gráfico visualiza el DiD, no prueba tendencias paralelas.',
        ha='center', transform=ax.transAxes, fontsize=9, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# --- Regresión DiD ---
datos['DxT'] = datos['D'] * datos['t']
mod_did = smf.ols('y ~ D + t + DxT', data=datos).fit(cov_type='HC1')
print('Regresión DiD:')
print(mod_did.summary().tables[1])
print(f'\n→ Coeficiente DxT (DiD): {mod_did.params["DxT"]:.4f}  (esperado ~{efecto_real})')

---
## PARTE 2: DiD CON MÚLTIPLES PERIODOS

**Datos simulados equivalentes a `webuse hospdd` (Stata)**  
- 50 hospitales observados durante 10 años (2001–2010)
- Adopción **escalonada**: cohortes que adoptan en 2005, 2006, 2007 o nunca
- Resultado: satisfacción de pacientes (0–100)
- Efecto real del programa: **+5 puntos**

Usamos `pyfixest` como equivalente de `xtdidregress` + `estat grangerplot` de Stata.  
Para las pruebas formales, usamos test F conjunto de los coeficientes pre-tratamiento.

| Stata | Python (pyfixest) |
|-------|-------------------|
| `xtdidregress` | `feols('y ~ treatment | hospital + time')` |
| `estat grangerplot` | `iplot()` o gráfico manual del event study |
| `estat ptrends` | Test F conjunto de coefs pre-tratamiento |
| `estat granger` | Mismo test, distinta interpretación conceptual |

In [ ]:
# --- Simulación de datos equivalentes a hospdd ---
np.random.seed(42)

n_hosp  = 50
years   = list(range(2001, 2011))
efecto_prog = 5

# Cohortes de adopción
cohorts = np.random.choice([2005, 2006, 2007, np.inf], n_hosp,
                            p=[0.3, 0.3, 0.2, 0.2])
efecto_hosp = np.random.normal(0, 8, n_hosp)  # heterogeneidad hospitalaria

filas = []
for h in range(n_hosp):
    for t in years:
        treat = int(t >= cohorts[h]) if np.isfinite(cohorts[h]) else 0
        y = (60 + efecto_hosp[h] +
             0.5 * (t - 2001) +
             efecto_prog * treat +
             np.random.normal(0, 3))
        filas.append({'hospital': h+1, 'time': t,
                      'treatment': treat,
                      'cohort': cohorts[h],
                      'satisfaction': y})

hospdd = pd.DataFrame(filas)
hospdd['cohort_label'] = hospdd['cohort'].apply(
    lambda x: str(int(x)) if np.isfinite(x) else 'Nunca'
)

print(f'Observaciones: {len(hospdd)}')
print(f'Efecto real: {efecto_prog} puntos de satisfacción')
print('\nDistribución de cohortes:')
print(hospdd.groupby('cohort_label')['hospital'].nunique())

In [ ]:
# --- Gráfico de tendencias por cohorte ---
medias_h = hospdd.groupby(['cohort_label', 'time'])['satisfaction'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
colores = {'2005': '#1C3A6E', '2006': '#D55E00', '2007': '#009E73', 'Nunca': '#999999'}

for cohorte, datos_c in medias_h.groupby('cohort_label'):
    ax.plot(datos_c['time'], datos_c['satisfaction'],
            color=colores.get(cohorte, 'black'),
            linewidth=2, marker='o', markersize=5, label=f'Adopta {cohorte}')

for yr in [2004.5, 2005.5, 2006.5]:
    ax.axvline(yr, color='gray', linestyle='--', linewidth=0.8)

ax.set_title('Satisfacción media por cohorte y año', fontsize=13)
ax.set_xlabel('Año')
ax.set_ylabel('Satisfacción media')
ax.legend(title='Grupo', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- TWFE: equivalente a xtdidregress ---
mod_twfe = pf.feols('satisfaction ~ treatment | hospital + time',
                    data=hospdd, vcov={'CRV1': 'hospital'})
print('TWFE (equivalente a xtdidregress):')
mod_twfe.summary()
coef_twfe = mod_twfe.coef()['treatment']
print(f'\n→ Coeficiente treatment: {coef_twfe:.3f}  (esperado ~{efecto_prog})')

In [ ]:
# --- EVENT STUDY: equivalente a estat grangerplot ---
# Construir tiempo relativo al año de adopción
hospdd['t_rel'] = hospdd.apply(
    lambda r: r['time'] - r['cohort'] if np.isfinite(r['cohort']) else -99, axis=1
).astype(int)
hospdd['t_rel_bin'] = hospdd['t_rel'].clip(-4, 4)
hospdd.loc[hospdd['cohort'].apply(lambda x: not np.isfinite(x)), 't_rel_bin'] = -99

# Regresión event study (referencia = -1 y grupo nunca-adoptante = -99)
# Crear dummies manualmente
periodos = sorted([x for x in hospdd['t_rel_bin'].unique() if x not in [-1, -99]])
for p in periodos:
    hospdd[f'rel_{p}'] = ((hospdd['t_rel_bin'] == p) & (hospdd['t_rel_bin'] != -99)).astype(int)

vars_es = [f'rel_{p}' for p in periodos]
formula_es = f'satisfaction ~ {"+".join(vars_es)} | hospital + time'

mod_es = pf.feols(formula_es, data=hospdd, vcov={'CRV1': 'hospital'})
print('Event study (equivalente a estat grangerplot):')
mod_es.summary()

In [ ]:
# --- Gráfico event study ---
coefs = mod_es.coef()
ses   = mod_es.se()

plot_data = pd.DataFrame({
    'periodo': periodos,
    'coef':    [coefs[f'rel_{p}'] for p in periodos],
    'se':      [ses[f'rel_{p}']   for p in periodos]
})
# Agregar el periodo de referencia (k=-1, coef=0)
plot_data = pd.concat([
    plot_data,
    pd.DataFrame({'periodo': [-1], 'coef': [0.0], 'se': [0.0]})
]).sort_values('periodo').reset_index(drop=True)

ci95 = 1.96 * plot_data['se']

fig, ax = plt.subplots(figsize=(9, 5))
pre_mask  = plot_data['periodo'] < 0
post_mask = plot_data['periodo'] >= 0

ax.errorbar(plot_data.loc[pre_mask,  'periodo'], plot_data.loc[pre_mask,  'coef'],
            yerr=ci95[pre_mask],  fmt='o', color='#8B1A1A', capsize=4, label='Pre-tratamiento')
ax.errorbar(plot_data.loc[post_mask, 'periodo'], plot_data.loc[post_mask, 'coef'],
            yerr=ci95[post_mask], fmt='s', color='#1C3A6E', capsize=4, label='Post-tratamiento')

ax.axhline(0,  color='black', linestyle='-',  linewidth=0.8)
ax.axvline(-0.5, color='gray', linestyle='--', linewidth=1.2, label='Inicio tratamiento')
ax.axhline(efecto_prog, color='green', linestyle=':', linewidth=1.2, label=f'Efecto real ({efecto_prog})')

ax.set_title('Event study: efecto por año relativo al tratamiento\n(equivalente a estat grangerplot)',
             fontsize=12)
ax.set_xlabel('Años relativos al inicio del tratamiento')
ax.set_ylabel('Efecto estimado en satisfacción')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('\nInterpretación:')
print('• Periodos pre (k<0): deben ser ≈ 0 si no hay anticipación ni pre-tendencias')
print('• Periodos post (k≥0): muestran la dinámica del efecto en el tiempo')

In [ ]:
# --- PRUEBA DE TENDENCIAS PARALELAS (equivalente a estat ptrends) ---
# Test F conjunto de coeficientes pre-tratamiento (k < 0, excluyendo referencia -1)
from scipy import stats

print('=== PRUEBA DE TENDENCIAS PARALELAS (equivalente a estat ptrends) ===')
print('H0: Los coeficientes pre-tratamiento son conjuntamente = 0')
print('    (las tendencias lineales eran paralelas antes del programa)\n')

vars_pre = [f'rel_{p}' for p in periodos if p < -1]  # excluye referencia (-1)
print('Coeficientes pre-tratamiento en la prueba:', vars_pre)

# Usar wald test de pyfixest
resultado_wald = mod_es.wald_test(vars_pre)
print(f'\nF-stat: {resultado_wald["statistic"]:.4f}')
print(f'p-valor: {resultado_wald["pvalue"]:.4f}')

if resultado_wald['pvalue'] > 0.05:
    print('→ NO rechazamos H0. Las tendencias pre-tratamiento son paralelas. ✓')
else:
    print('→ Rechazamos H0. Hay evidencia de tendencias no paralelas.')

In [ ]:
# --- PRUEBA DE ANTICIPACIÓN (equivalente a estat granger) ---
print('=== PRUEBA DE ANTICIPACIÓN (equivalente a estat granger) ===')
print('H0: Sin efectos del programa ANTES de que empezara')
print()
print('Mecánica: mismos coeficientes pre-tratamiento, mismo test F.')
print('La diferencia entre ptrends y granger es CONCEPTUAL, no estadística:')
print()
print('┌─────────────────┬───────────────────────────────────┬───────────────────────────��─────────┐')
print('│ Prueba          │ H0                                │ Si se rechaza...                    │')
print('├─────────────────┼───────────────────────────────────┼─────────────────────────────────────┤')
print('│ ptrends         │ Tendencias lineales iguales pre-t │ Grupos con pendientes distintas     │')
print('│ granger         │ Sin efectos pre-tratamiento       │ Anticipación O tendencias distintas │')
print('└─────────────────┴───────────────────────────────────┴─────────────────────────────────────┘')
print()
print('Son observacionalmente equivalentes. Distinguirlos requiere argumento teórico.')
print()
print('RESTRICCIÓN: Ambas pruebas necesitan ≥ 2 periodos PRE-tratamiento.')
print('Con solo 2 periodos (antes/después), estas pruebas no son posibles.')

---
## Resumen

| Comando Stata | Equivalente Python | ¿Qué hace? |
|---------------|-------------------|------------|
| `xtdidregress (y) (trat), group(id) time(t)` | `feols('y ~ trat \| id + t')` | DiD con efectos fijos de individuo y tiempo |
| `estat trendplots` | Gráfico de medias por cohorte | Diagnóstico visual de tendencias |
| `estat ptrends` | Wald test de coefs pre-tratamiento | Prueba de **tendencias paralelas** |
| `estat granger` | Mismo test, distinta interpretación | Prueba de **anticipación** |
| `estat grangerplot` | `iplot()` o gráfico manual | Event study: efectos por periodo |

**Diferencia clave entre `ptrends` y `granger`:** son el mismo test estadístico pero responden preguntas distintas. Para saber cuál aplica en tu caso, necesitas un argumento teórico sobre el contexto del programa.